# Análisis de fragmentación de IDs — StrongKalmanTrackerV3

Diagnostica por qué los tracks se fragmentan y cómo mejorar la Re-ID.

## Requisitos externos
- **Vídeo panorámico VEO** (`data/videos/veo_panoramico_banyoles.mp4`) — descargar con `yt-dlp`
- **Modelo YOLO jugadores** (`runs/detect/modelo_players_v24_panoramic2/weights/best.pt`) — entrenar con `finetune_players_panoramic.py`
- **Homografías y prototipos de ejemplo** incluidos en `data/example_banyoles/` (campo Banyoles 98×61 m)

In [ ]:
import sys, time, threading, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2, pickle, os
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
import pipeline_core as core
from ultralytics import YOLO
print('OK')

In [ ]:
VIDEO_PATH  = 'data/videos/veo_panoramico_banyoles.mp4'    # descargar con yt-dlp
H_A_PATH    = 'data/example_banyoles/H_camA.npy'
H_B_PATH    = 'data/example_banyoles/H_camB.npy'
PKL_DAY     = 'data/example_banyoles/prototypes_day.pkl'
PKL_NIGHT   = 'data/example_banyoles/prototypes_night.pkl'
MODEL_PATH  = 'runs/detect/modelo_players_v24_panoramic2/weights/best.pt'
START_FRAME = 18000
N_FRAMES    = 900
FPS         = 30.0
CONFIG = {
    'start_frame': START_FRAME, 'n_frames': N_FRAMES,
    'tracker_type': 'strongkalman',
    'conf_thresh': 0.35, 'conf_ball': 0.70,
    'bright_thresh': 75.0, 'field_length': 98.0, 'field_width': 61.0,
    'home_attacks_left': True, 'halftime_frame': None,
    'match_intervals': None, 'gk_home_color_lab': None, 'gk_away_color_lab': None,
    'track_max_dist_m': 8.0, 'track_window_s': 7.0,
    'track_class_w': 3.0, 'v3_lost_s': 7.0,
    'device': 'cuda', 'half_precision': True, 'imgsz': 1280,
}
FL = CONFIG['field_length']
FW = CONFIG['field_width']
print('Config OK')

In [3]:
H_a = np.load(H_A_PATH)
H_b = np.load(H_B_PATH)
protos_day   = pickle.load(open(PKL_DAY, 'rb'))
protos_night = pickle.load(open(PKL_NIGHT, 'rb')) if os.path.exists(PKL_NIGHT) else None
model_A = YOLO(MODEL_PATH)
model_B = YOLO(MODEL_PATH)
print('Cargado OK')


Cargado OK


In [4]:
progress = {}
t0 = time.perf_counter()
thread = threading.Thread(
    target=core.process_segment,
    args=(VIDEO_PATH, None, H_a, H_b,
          protos_day, protos_night, CONFIG,
          model_A, model_B, None, progress, None),
    daemon=True
)
thread.start()
while not progress.get('done') and not progress.get('error'):
    print('\r{:.0f}%  '.format(progress.get('pct', 0)), end='', flush=True)
    time.sleep(0.5)
elapsed = time.perf_counter() - t0
if progress.get('error'):
    print('ERROR:', progress['error'][:300])
else:
    records = progress.get('frame_records', [])
    df = pd.DataFrame(records)
    print('\nDone {:.1f}s  ({:.1f} fps)  filas={}  IDs={}'.format(
        elapsed, N_FRAMES/elapsed, len(df), df['gid'].nunique()))


1%  
Done 371.1s  (2.4 fps)  filas=16468  IDs=28


In [5]:
lengths_s = df.groupby('gid')['frame'].count() / FPS
total = len(lengths_s)
print('Total IDs: ', total)
print('Mediana:   {:.2f}s'.format(lengths_s.median()))
print('Media:     {:.2f}s'.format(lengths_s.mean()))
print('Max:       {:.2f}s'.format(lengths_s.max()))
print('')
for t in [0.5, 1.0, 2.0, 5.0, 10.0, 20.0]:
    n = int((lengths_s >= t).sum())
    print('>= {:<5.1f}s  {:>4d}  ({:.0f}%)'.format(t, n, 100*n/total))
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(lengths_s, bins=60, color='#1E90FF', edgecolor='none', alpha=0.85)
for t, lbl in [(1.0, '1s'), (5.0, '5s'), (10.0, '10s')]:
    ax.axvline(t, color='red', lw=1.2, ls='--', label=lbl)
ax.set_xlabel('Duracion track (s)')
ax.set_ylabel('N IDs')
ax.set_title('Distribucion de duracion de tracks')
ax.legend()
plt.tight_layout(); plt.show()


Total IDs:  28
Mediana:   23.62s
Media:     19.60s
Max:       29.90s

>= 0.5  s    27  (96%)
>= 1.0  s    27  (96%)
>= 2.0  s    27  (96%)
>= 5.0  s    23  (82%)
>= 10.0 s    22  (79%)
>= 20.0 s    18  (64%)


In [6]:
active = df.groupby('frame')['gid'].nunique().reindex(
    range(START_FRAME, START_FRAME+N_FRAMES), fill_value=0).reset_index()
active.columns = ['frame', 'n_active']
active['t_s'] = (active['frame'] - START_FRAME) / FPS
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(active['t_s'], active['n_active'], lw=1, color='#1E90FF')
ax.axhline(22, color='red', lw=0.8, ls='--', label='22 esperados')
ax.fill_between(active['t_s'], active['n_active'], alpha=0.15, color='#1E90FF')
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('IDs activos')
ax.set_title('Detecciones activas por frame')
ax.legend()
plt.tight_layout(); plt.show()
print('Media activos/frame: {:.1f}  max: {}  min: {}'.format(
    active['n_active'].mean(), active['n_active'].max(), active['n_active'].min()))


Media activos/frame: 18.3  max: 22  min: 0


In [7]:
short_ids = lengths_s[lengths_s < 2.0].index
long_ids  = lengths_s[lengths_s >= 5.0].index
first_pos = df.sort_values('frame').groupby('gid').first()[['x_m','y_m']]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ids, title, color in [
    (axes[0], short_ids, 'Inicio tracks cortos (<2s)', '#FF4444'),
    (axes[1], long_ids,  'Inicio tracks largos (>=5s)', '#44BB44'),
]:
    ax.set_facecolor('#1a472a')
    ax.set_xlim(0, FL); ax.set_ylim(0, FW); ax.set_aspect('equal')
    for lx in [FL/4, FL/2, 3*FL/4]: ax.axvline(lx, color='white', lw=0.4, alpha=0.3)
    ax.axhline(FW/2, color='white', lw=0.4, alpha=0.3)
    pos = first_pos.loc[first_pos.index.intersection(ids)]
    ax.scatter(pos['x_m'], pos['y_m'], s=14, c=color, alpha=0.6, linewidths=0)
    ax.set_title('{} (n={})'.format(title, len(pos)))
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.tight_layout(); plt.show()


In [8]:
gap_rows = []
for gid, grp in df.groupby('gid'):
    frames = np.array(sorted(grp['frame'].unique()))
    if len(frames) < 2: continue
    diffs = np.diff(frames)
    gap_rows.append({'gid': gid, 'max_gap_f': int(diffs.max()),
                     'max_gap_s': diffs.max()/FPS, 'len_s': len(frames)/FPS})
gdf = pd.DataFrame(gap_rows)
for th in [3, 5, 10, 20, 30]:
    n = int((gdf['max_gap_f'] > th).sum())
    print('Tracks con gap > {} frames: {} ({:.0f}%)'.format(th, n, 100*n/len(gdf)))
fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(gdf['len_s'], gdf['max_gap_s'], s=10, alpha=0.4, color='#FF8C00')
ax.axhline(CONFIG['v3_lost_s'], color='red', lw=1.2, ls='--',
           label='v3_lost_s={:.1f}s'.format(CONFIG['v3_lost_s']))
ax.set_xlabel('Duracion track (s)')
ax.set_ylabel('Max gap dentro del track (s)')
ax.set_title('Gaps de deteccion vs duracion de track')
ax.legend(); plt.tight_layout(); plt.show()


Tracks con gap > 3 frames: 23 (82%)
Tracks con gap > 5 frames: 21 (75%)
Tracks con gap > 10 frames: 17 (61%)
Tracks con gap > 20 frames: 13 (46%)
Tracks con gap > 30 frames: 9 (32%)


In [9]:
mode_cls = df.groupby('gid')['cls'].agg(lambda x: x.mode().iloc[0])
len_cls  = pd.DataFrame({'len_s': lengths_s, 'cls': mode_cls})
CLASS_COLORS = {
    'player_home': '#DC143C', 'player_away': '#1E90FF',
    'gk_home': '#FF00FF', 'gk_away': '#00CED1',
    'referee': '#FFD700', 'unknown': '#888888',
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mask, title in [
    (axes[0], len_cls['len_s'] < 2.0,  'Tracks cortos (<2s)'),
    (axes[1], len_cls['len_s'] >= 5.0, 'Tracks largos (>=5s)'),
]:
    counts = len_cls[mask]['cls'].value_counts()
    if len(counts) == 0: ax.text(0.5, 0.5, 'Sin datos', ha='center'); continue
    colors = [CLASS_COLORS.get(c, '#aaa') for c in counts.index]
    ax.pie(counts.values, labels=counts.index, colors=colors,
           autopct='%1.0f%%', textprops={'fontsize': 9})
    ax.set_title('{} (n={})'.format(title, int(mask.sum())))
plt.suptitle('Clase segun duracion del track')
plt.tight_layout(); plt.show()


In [10]:
top25 = lengths_s.nlargest(25).index
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
cmap = plt.cm.tab20
for i, gid in enumerate(top25):
    grp = df[df['gid']==gid].sort_values('frame')
    t = (grp['frame'] - START_FRAME) / FPS
    c = cmap(i/25)
    axes[0].plot(t, grp['x_m'], lw=1, alpha=0.8, color=c,
                 label='g{} ({:.1f}s)'.format(gid, lengths_s[gid]))
    axes[1].plot(t, grp['y_m'], lw=1, alpha=0.8, color=c)
axes[0].set_ylabel('x (m)'); axes[0].set_title('Top-25 tracks: coordenada X')
axes[0].legend(fontsize=5.5, ncol=5, loc='upper right')
axes[1].set_ylabel('y (m)'); axes[1].set_xlabel('Tiempo (s)')
axes[1].set_title('Top-25 tracks: coordenada Y')
plt.tight_layout(); plt.show()


In [11]:
frac_short  = (lengths_s < 2.0).mean()
frac_long   = (lengths_s >= 10.0).mean()
mean_active = active['n_active'].mean()
print('=' * 52)
print('RESUMEN DIAGNOSTICO')
print('=' * 52)
print('Total IDs:              ', total)
print('Tracks < 2s:        {:5.1f}%  ({})'.format(100*frac_short,  int(frac_short*total)))
print('Tracks >= 10s:      {:5.1f}%  ({})'.format(100*frac_long,   int(frac_long*total)))
print('Media activos/frame: {:.1f}  (esperado ~22)'.format(mean_active))
print('')
print('Parametros actuales:')
for k in ['conf_thresh', 'v3_lost_s', 'track_max_dist_m', 'track_class_w']:
    print('  {:20s} = {}'.format(k, CONFIG[k]))
print('')
if frac_short > 0.4:
    print('[!] Muchos tracks cortos. Causas probables y soluciones:')
    print('  1. conf_thresh=0.52 alto -> prueba 0.40-0.45')
    print('  2. track_max_dist_m=6 pequenyo -> prueba 8.0 o 10.0')
    print('  3. v3_lost_s=4 insuficiente para oclusiones -> prueba 6.0')
    print('  4. class_weight=8 muy alto -> penaliza reasignacion -> prueba 4.0-6.0')
if mean_active < 15:
    print('[!] Pocas detecciones activas ({:.1f}/frame) -> baja conf_thresh'.format(mean_active))


RESUMEN DIAGNOSTICO
Total IDs:               28
Tracks < 2s:          3.6%  (1)
Tracks >= 10s:       78.6%  (22)
Media activos/frame: 18.3  (esperado ~22)

Parametros actuales:
  conf_thresh          = 0.35
  v3_lost_s            = 7.0
  track_max_dist_m     = 8.0
  track_class_w        = 3.0

